# 02 - Control Flow & Functions

**Sources:** Bootcamp BP2 (control flow, `match`, loop `else`, functions, parameters, annotations,
scopes, lambdas) and Summer School *CursP_BasePython* (conditionals, loops, exceptions, defining
functions, mutability of arguments, global variables, `*args/**kwargs`, docstrings, first-class
functions).

**Contents:** conditionals -> loops -> `match` -> exceptions -> functions -> arguments ->
scope & closures -> lambda & functional tools -> recursion.

## 1. Conditionals
Blocks are delimited by **indentation**; conditions need no parentheses.

In [1]:
a = 10
if a == 1:
    print(1)
elif a == 2:
    print(2)
else:
    print("A lot")

A lot


In [2]:
# Conditional expression (ternary)
age = 20
status = "adult" if age >= 18 else "minor"
print(status)

# Chained comparisons
x = 5
print(0 < x < 10, 1 == 1.0, "a" < "b" < "c")

# Truthiness drives `if`: an empty container is False
items = []
if not items:
    print("nothing to do")

adult
True True True
nothing to do


## 2. Loops
`for` iterates over **any iterable** - you almost never need an index.

In [3]:
for word in ("cool", "powerful", "readable"):
    print(f"Python is {word}")

for i in range(4):          # 0, 1, 2, 3
    print(i, end=" ")
print()
print(list(range(10, 0, -3)), list(range(1, 10, 4)))

Python is cool
Python is powerful
Python is readable
0 1 2 3 
[10, 7, 4, 1] [1, 5, 9]


In [4]:
# enumerate (index + value), zip (parallel iteration), reversed, sorted
words = ("cool", "powerful", "readable")
for index, item in enumerate(words, start=1):
    print(index, item)

for w, n in zip(words, [4, 8, 8]):
    print(w, n)

for ch in reversed("abc"):
    print(ch, end="")
print()

1 cool
2 powerful
3 readable
cool 4
powerful 8
readable 8
cba


In [5]:
# Iterating dicts
d = {"a": 1, "b": 1.2, "c": 1j}
for key, val in d.items():          # Python 2's iteritems() is gone
    print(f"Key: {key} has value: {val}")

Key: a has value: 1
Key: b has value: 1.2
Key: c has value: 1j


In [6]:
# while / break / continue / pass
z = 1 + 1j
steps = 0
while abs(z) < 100:                 # Mandelbrot-style escape loop
    z = z ** 2 + 1
    steps += 1
print("escaped after", steps, "steps")

for element in [1, 0, 2, 4]:
    if element == 0:
        continue                     # skip to the next iteration
    print(1.0 / element, end=" ")
print()

for n in range(100):
    if n * n > 50:
        print("first n with n^2 > 50:", n)
        break

def todo():
    pass                             # placeholder - does nothing

escaped after 4 steps
1.0 0.5 0.25 
first n with n^2 > 50: 8


### The versatile `else` clause
* `for/while ... else`: runs when the loop ended **without** `break`.
* `try ... else`: runs when the `try` body raised **no** exception.

In [7]:
def is_prime(n):
    if n < 2:
        return False
    for d in range(2, int(n ** 0.5) + 1):
        if n % d == 0:
            break
    else:                            # no divisor found
        return True
    return False

print([n for n in range(30) if is_prime(n)])

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


## 3. Structural pattern matching: `match` (3.10+)
Like `switch`, but it matches **shapes**: literals, sequences, mappings, class instances, guards.

In [8]:
def describe(obj):
    match obj:
        case 0 | 1:
            return "bit"
        case int(n) if n < 0:
            return f"negative int {n}"
        case int() | float():
            return "number"
        case str() as s if s.isdigit():
            return f"digit-string {s}"
        case [x, y]:
            return f"pair {x},{y}"
        case [first, *rest]:
            return f"sequence starting with {first} (+{len(rest)} more)"
        case {"type": "circle", "r": r}:
            return f"circle radius {r}"
        case {"type": kind}:
            return f"unknown shape {kind}"
        case None:
            return "nothing"
        case _:
            return "something else"

for sample in [1, -5, 3.14, "42", (1, 2), [1, 2, 3, 4], {"type": "circle", "r": 2}, {"type": "box"}, None, object()]:
    print(f"{sample!r:>32} -> {describe(sample)}")

                               1 -> bit
                              -5 -> negative int -5
                            3.14 -> number
                            '42' -> digit-string 42
                          (1, 2) -> pair 1,2
                    [1, 2, 3, 4] -> sequence starting with 1 (+3 more)
      {'type': 'circle', 'r': 2} -> circle radius 2
                 {'type': 'box'} -> unknown shape box
                            None -> nothing
<object object at 0x00000161A25EB8D0> -> something else


In [9]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

def where(p):
    match p:
        case Point(x=0, y=0):
            return "origin"
        case Point(x=0):
            return "on the y axis"
        case Point(y=0):
            return "on the x axis"
        case Point(x=x, y=y):
            return f"at ({x}, {y})"

print(where(Point(0, 0)), where(Point(0, 3)), where(Point(2, 0)), where(Point(1, 1)))

origin on the y axis on the x axis at (1, 1)


## 4. Exceptions

Errors are objects. `try / except / else / finally`:
* `except` - handle specific exceptions (most specific first, never a bare `except:`)
* `else` - success path
* `finally` - **always** runs: clean-up (close files, release locks)

In [10]:
errors = [lambda: 1 / 0, lambda: 1 + "e", lambda: {1: 1}[3], lambda: [1, 2, 3][4],
          lambda: [].foobar, lambda: int("abc")]
for fn in errors:
    try:
        fn()
    except Exception as err:         # demo only - normally catch specific types
        print(f"{type(err).__name__:>18}: {err}")

 ZeroDivisionError: division by zero
         TypeError: unsupported operand type(s) for +: 'int' and 'str'
          KeyError: 3
        IndexError: list index out of range
    AttributeError: 'list' object has no attribute 'foobar'
        ValueError: invalid literal for int() with base 10: 'abc'


In [11]:
def parse_positive(text):
    try:
        value = int(text)
    except ValueError:
        print(f"  {text!r} is not a number")
        return None
    else:
        print(f"  parsed {value}")
        return value
    finally:
        print("  (finally always runs)")

parse_positive("12"); parse_positive("abc")

  parsed 12
  (finally always runs)
  'abc' is not a number
  (finally always runs)


In [12]:
# Raising and custom exceptions; chaining with `from`
class NegativeError(ValueError):
    """Raised when a non-negative value is required."""

def sqrt_int(n):
    if n < 0:
        raise NegativeError(f"expected n >= 0, got {n}")
    return int(n ** 0.5)

try:
    sqrt_int(-4)
except NegativeError as err:
    print("caught:", err, "| is ValueError too:", isinstance(err, ValueError))

try:
    try:
        int("x")
    except ValueError as err:
        raise RuntimeError("config is broken") from err
except RuntimeError as err:
    print(repr(err), "<- caused by", repr(err.__cause__))

caught: expected n >= 0, got -4 | is ValueError too: True
RuntimeError('config is broken') <- caused by ValueError("invalid literal for int() with base 10: 'x'")


In [13]:
# The classic input loop from the slides (non-interactive version)
attempts = iter(["a", "1.5", "7"])
while True:
    raw = next(attempts)
    try:
        x = int(raw)
        break
    except ValueError:
        print("That was no valid number. Try again...")
print("got", x)

# EAFP ("easier to ask forgiveness than permission") is idiomatic:
data = {"a": 1}
try:
    value = data["b"]
except KeyError:
    value = 0
print(value)

That was no valid number. Try again...
That was no valid number. Try again...
got 7
0


## 5. Defining functions
`def name(params): body` - returns `None` unless `return` is given. Functions are **objects**
(first-class): you can assign them, store them in containers, and pass them around.

In [14]:
def disk_area(radius):
    """Return the area of a disk. (First line = concise summary, per PEP 257.)"""
    return 3.14159 * radius * radius

print(disk_area(1.5))
print(disk_area.__name__, "|", disk_area.__doc__)
help(disk_area)

7.068577499999999
disk_area | Return the area of a disk. (First line = concise summary, per PEP 257.)
Help on function disk_area in module __main__:

disk_area(radius)
    Return the area of a disk. (First line = concise summary, per PEP 257.)



### Parameter kinds

In [15]:
def demo(pos_only, /, normal, *, kw_only, default=2):
    return pos_only, normal, kw_only, default

print(demo(1, 2, kw_only=3))
print(demo(1, normal=2, kw_only=3, default=4))
try:
    demo(1, 2, 3)
except TypeError as err:
    print("TypeError:", err)

(1, 2, 3, 2)
(1, 2, 3, 4)
TypeError: demo() takes 2 positional arguments but 3 were given


In [16]:
# *args collects extra positionals into a tuple, **kwargs extra keywords into a dict
def variable_args(*args, **kwargs):
    print("args  is", args)
    print("kwargs is", kwargs)

variable_args("one", "two", x=1, y=2, z=3)

# Unpacking at the call site is the mirror image
values = [1, 2, 3]
options = {"sep": "-", "end": "!\n"}
print(*values, **options)

args  is ('one', 'two')
kwargs is {'x': 1, 'y': 2, 'z': 3}
1-2-3!


In [17]:
# The slicer example from the slides
def slicer(seq, start=None, stop=None, step=None):
    """Implement basic python slicing."""
    return seq[start:stop:step]

rhyme = "one fish, two fish, red fish, blue fish".split()
print(slicer(rhyme))
print(slicer(rhyme, step=2))
print(slicer(rhyme, 1, step=2))
print(slicer(rhyme, start=1, stop=4, step=2))

['one', 'fish,', 'two', 'fish,', 'red', 'fish,', 'blue', 'fish']
['one', 'two', 'red', 'blue']
['fish,', 'fish,', 'fish,', 'fish']
['fish,', 'fish,']


### Default values: evaluated ONCE, at definition time
This bites in two ways (both are in the slides/bootcamp):

In [18]:
bigx = 10
def double_it(x=bigx):
    return x * 2

bigx = 1e9
print(double_it())            # still 20, not 2e9: the default was frozen at `def` time

20


In [19]:
def append_bad(item, bucket=[]):          # DANGER: one shared list for every call
    bucket.append(item)
    return bucket

print(append_bad(1), append_bad(2))       # [1] then [1, 2]  <- surprise

def append_good(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(append_good(1), append_good(2))
print(append_bad.__defaults__)            # introspection shows the leaked state

[1, 2] [1, 2]
[1] [2]
([1, 2],)


### Annotations (type hints)
Metadata only - Python does not enforce them, but tools (mypy, IDEs) do. More in notebook 04.

In [20]:
def scale(values: list[float], factor: float = 2.0) -> list[float]:
    return [v * factor for v in values]

print(scale([1, 2, 3]))
print(scale.__annotations__)

[2.0, 4.0, 6.0]
{'values': list[float], 'factor': <class 'float'>, 'return': list[float]}


## 6. Arguments are passed "by object reference"

The function receives a *reference to the same object*. **Mutating** a mutable argument is visible
to the caller; **rebinding** the parameter name is not.

In [21]:
def try_to_modify(x, y, z):
    x = 23            # rebinding an immutable -> local only
    y.append(42)      # mutation -> visible outside
    z = [99]          # rebinding to a NEW list -> local only
    print("inside :", x, y, z)

a = 77
b = [99]
c = [28]
try_to_modify(a, b, c)
print("outside:", a, b, c)

inside : 23 [99, 42] [99]
outside: 77 [99, 42] [28]


## 7. Scope: LEGB
Name lookup order: **L**ocal -> **E**nclosing function -> **G**lobal (module) -> **B**uilt-in.
* assignment inside a function creates a **local** unless declared `global` / `nonlocal`.

In [22]:
x = 5

def add_x(y):
    return x + y                  # reads the global

def set_x_local(y):
    x = y                         # creates a LOCAL x; global untouched
    return x

def set_x_global(y):
    global x
    x = y

print(add_x(10), set_x_local(99), x)
set_x_global(10)
print(x)

15 99 5
10


In [23]:
# Closures: an inner function remembers variables of the enclosing function
def make_counter(start=0):
    count = start
    def tick():
        nonlocal count            # rebind the enclosing variable
        count += 1
        return count
    return tick

c1, c2 = make_counter(), make_counter(100)
print(c1(), c1(), c2(), c1())

# Late-binding gotcha: closures capture the VARIABLE, not its value at creation time
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])                       # [2, 2, 2]
funcs = [lambda i=i: i for i in range(3)]         # bind the current value as a default
print([f() for f in funcs])                       # [0, 1, 2]

1 2 101 3
[2, 2, 2]
[0, 1, 2]


## 8. Lambdas and functional tools
`lambda args: expression` - an anonymous single-expression function; handy as a `key=`.

In [24]:
square = lambda v: v * v
print(square(7))

people = [("ann", 31), ("bob", 25), ("cy", 40)]
print(sorted(people, key=lambda p: p[1]))
print(max(people, key=lambda p: p[1]))
print(sorted(["banana", "Apple", "cherry"], key=str.lower))

print(list(map(lambda v: v ** 2, range(5))), [v ** 2 for v in range(5)])   # comprehension is usually clearer
print(list(filter(lambda v: v % 2, range(10))))

from functools import reduce, partial
print(reduce(lambda acc, v: acc * v, range(1, 6)))     # reduce moved to functools in Python 3
power_of_two = partial(pow, 2)
print(power_of_two(10))

# Functions are first-class
operations = {"double": lambda v: v * 2, "negate": lambda v: -v, "square": square}
for name, fn in operations.items():
    print(name, fn(6))

49
[('bob', 25), ('ann', 31), ('cy', 40)]
('cy', 40)
['Apple', 'banana', 'cherry']
[0, 1, 4, 9, 16] [0, 1, 4, 9, 16]
[1, 3, 5, 7, 9]
120
1024
double 12
negate -6
square 36


## 9. Recursion
A function that calls itself needs a **base case**. CPython limits recursion depth (~1000) and has
no tail-call optimisation, so deep recursion -> iterate or use a stack.

In [25]:
import sys
from functools import lru_cache

def factorial(n):
    return 1 if n <= 1 else n * factorial(n - 1)

def fib_slow(n):                    # exponential time - see Codility lesson 13
    return n if n < 2 else fib_slow(n - 1) + fib_slow(n - 2)

@lru_cache(maxsize=None)            # memoisation turns it into linear time
def fib_fast(n):
    return n if n < 2 else fib_fast(n - 1) + fib_fast(n - 2)

print(factorial(10), fib_slow(20), fib_fast(80))
print("recursion limit:", sys.getrecursionlimit())

3628800 6765 23416728348467685
recursion limit: 1000


## Exercises (from the Summer School slides)

**A. Fibonacci (10 min).** Write a function that returns the first `n` terms of
`u0 = 1, u1 = 1, u(n+2) = u(n+1) + u(n)`. Do it iteratively *and* with a generator (see notebook 04).

In [26]:
def fibonacci_terms(n):
    """Return a list with the first n terms."""
    raise NotImplementedError

# fibonacci_terms(8)  ->  [1, 1, 2, 3, 5, 8, 13, 21]

**B. Quicksort (15 min).** Implement quicksort recursively (pick a pivot, partition into smaller /
equal / larger, recurse). Compare it with `sorted` on random data using `random.sample`.

In [27]:
def quicksort(values):
    raise NotImplementedError

**C. Parameters and mutability (5 min).** Predict, then verify: what do these print?
```python
def f(a, b=[]): b.append(a); return b
print(f(1), f(2), f(3, [9]), f(4))
```

**D. Retry decorator-in-waiting.** Write `retry(func, attempts=3)` that calls `func()` and retries
when it raises `ConnectionError`, re-raising after the last attempt. (You will turn it into a real
decorator in notebook 04.)

In [28]:
def retry(func, attempts=3):
    raise NotImplementedError